# Final experiments, part 2: is it what the model can see, or how it is trained?

Part 1 showed that label history is the GNN's biggest single gain on trial outcomes (+0.060 AUROC without text, +0.034 with text), and that the GNN overfits. This notebook separates two explanations:

| Step | Run | Question |
|---|---|---|
| A | **RelGNN** (official per-task config), GloVe text | RelGNN's composite message passing skips junction tables, so a trial reaches its sponsors, conditions and sites in one layer instead of two. Does that recover part of what label history gives? |
| B | **RelGNN + label history** | Does history still help a model that sees further? |
| C | **GraphSAGE GNN, tuned** (12 random-search trials on validation: learning rate, width, dropout, weight decay, neighbors) | How much of the gap to LightGBM is just tuning? Same protocol as LightGBM's tuning. |
| D | **GraphSAGE GNN + label history, tuned** | Does the history gain survive tuning? |

All runs: both tasks, 3 seeds, GloVe text. **About 5–6 hours** on an L4; Colab Pro+ keeps it running if you close the tab. Results are added to `results_final/` on Drive, next to part 1.

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
RESULTS = f"{WORK}/results_final"
CACHE = f"{WORK}/cache"
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, torch_geometric, relbench, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

Tests (about 1–2 minutes), now including RelGNN and the tuning loop. **Expect 21 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

Bring in the earlier runs (milestone LightGBM + part 1) from the repo, so the final table has every model.

In [ ]:
for kind in ("runs", "preds"):
    for src in glob.glob(f"results/final/{kind}/*/*"):
        dst = src.replace("results/final", RESULTS)
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
print(sorted({p.split('/')[-1] for p in glob.glob(f"{RESULTS}/runs/*/*")}))

## Step A: RelGNN with text (about 1.5 hours)

In [ ]:
!python scripts/run.py rdl --arch relgnn --text glove --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Step B: RelGNN with text + label history (about 1.5 hours)

In [ ]:
!python scripts/run.py rdl --arch relgnn --text glove --history --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Step C: GraphSAGE GNN with text, tuned (about 1.5 hours)
12 random-search trials on validation (seed 0, early stopping after 6 epochs without improvement), then 3 seeds with the best settings.

In [ ]:
!python scripts/run.py rdl --text glove --tune 12 --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Step D: GraphSAGE GNN with text + label history, tuned (about 1.5 hours)

In [ ]:
!python scripts/run.py rdl --text glove --history --tune 12 --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Results table and download

In [ ]:
!python scripts/run.py tables --results {RESULTS}
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/tables/main.md").read()))

In [ ]:
shutil.make_archive("/content/final_results_part2", "zip", RESULTS, ".")
from google.colab import files
files.download("/content/final_results_part2.zip")